# Render a partitioned point cloud with Datashader

When many points fall on the same pixel, a scatter plot can hide dense regions. Count points per pixel across a Dask DataFrame, then color the resulting array. This example generates its own data and runs locally with Dask DataFrame, NumPy, pandas and Datashader installed.

In [ ]:
import dask.dataframe as dd
import datashader as ds
import datashader.transfer_functions as tf
import numpy as np
import pandas as pd
from dask import delayed

Each partition generates two clusters with different spreads. A seed per partition keeps the example reproducible. The delayed calls describe the work without first building the whole dataset in memory.

In [ ]:
@delayed
def make_partition(seed, size=25000):
    rng = np.random.default_rng(seed)
    centers = rng.choice([-2.0, 2.0], size=size)
    return pd.DataFrame({
        "x": centers + rng.normal(0, 0.55, size),
        "y": rng.normal(0, np.where(centers < 0, 0.45, 1.15)),
    })

meta = pd.DataFrame({"x": pd.Series(dtype="float64"), "y": pd.Series(dtype="float64")})
points = dd.from_delayed([make_partition(seed) for seed in range(8)], meta=meta)

Give the canvas explicit bounds so Datashader can aggregate without a preliminary scan for coordinate limits. Each partition contributes counts to the same pixel grid. Only the combined grid is needed for shading.

In [ ]:
canvas = ds.Canvas(plot_width=480, plot_height=320, x_range=(-6, 6), y_range=(-6, 6))
counts = canvas.points(points, "x", "y", agg=ds.count())
print(f"{points.npartitions} partitions; {int(counts.sum()):,} points inside the canvas")
tf.shade(counts, cmap=["#d1e9ed", "#367589", "#102f48"], how="log")

Both clusters contain roughly the same number of points. The tighter cluster on the left has more observations per occupied pixel. Logarithmic shading keeps both low-density edges and dense centers visible. To explore how resolution affects the counts, halve the canvas dimensions and render again.